# Model 03 — NBA Draft Cohort Analysis
**Discipline:** Growth DS  
**Question:** Do some NBA draft classes produce sustained value over time — and can cohort analysis reveal which draft years were genuinely exceptional vs just lucky?  
**Method:** Cohort retention analysis + heatmap (SaaS retention grid format) + ANOVA  
**Data:** NBA Advanced Stats via `nba_api` — draft classes 1990–2018, seasons 1990–91 through 2022–23

## 0. Imports & Setup

In [2]:
import time, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import f_oneway, kruskal, pearsonr, spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

OUTPUTS_DIR  = 'outputs/'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

DRAFT_YEARS  = list(range(1990, 2019))   # 1990–2018 draft classes
MAX_CAREER   = 15                         # cap career tracking at year 15
MIN_GP       = 20                         # min games to count as 'active' that season

print('Imports OK')

Imports OK


## 1. Data Collection — Draft History
Pull all NBA draft picks via `DraftHistory` endpoint. Gives us player_id, draft year, round, and pick number.

In [3]:
from nba_api.stats.endpoints import drafthistory

print('Fetching draft history...')
time.sleep(1)
dh_raw = drafthistory.DraftHistory(
    league_id='00',
    season_year_nullable=None,
    topx_nullable=None
).get_data_frames()[0]

print(f'Total records: {len(dh_raw)} | Columns: {dh_raw.columns.tolist()}')

dh_raw['DRAFT_YEAR'] = dh_raw['SEASON'].astype(int)
draft = (
    dh_raw[dh_raw['DRAFT_YEAR'].between(1990, 2018)]
    [['PERSON_ID', 'PLAYER_NAME', 'DRAFT_YEAR', 'ROUND_NUMBER', 'OVERALL_PICK']]
    .copy()
)
draft['PERSON_ID']     = draft['PERSON_ID'].astype('int64')
draft['ROUND_NUMBER']  = pd.to_numeric(draft['ROUND_NUMBER'],  errors='coerce').fillna(3).astype(int)
draft['OVERALL_PICK']  = pd.to_numeric(draft['OVERALL_PICK'],  errors='coerce').fillna(99).astype(int)

cohort_sizes = draft.groupby('DRAFT_YEAR').size().rename('COHORT_SIZE')
print(f'\nDraft picks 1990-2018: {len(draft)}')
print(f'Avg class size: {cohort_sizes.mean():.0f}')
draft.head()

Fetching draft history...
Total records: 8374 | Columns: ['PERSON_ID', 'PLAYER_NAME', 'SEASON', 'ROUND_NUMBER', 'ROUND_PICK', 'OVERALL_PICK', 'DRAFT_TYPE', 'TEAM_ID', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'ORGANIZATION', 'ORGANIZATION_TYPE', 'PLAYER_PROFILE_FLAG']

Draft picks 1990-2018: 1688
Avg class size: 58


,PERSON_ID,PLAYER_NAME,DRAFT_YEAR,ROUND_NUMBER,OVERALL_PICK
413,1629028,Deandre Ayton,2018,1,1
414,1628963,Marvin Bagley III,2018,1,2
415,1629029,Luka Dončić,2018,1,3
416,1628991,Jaren Jackson Jr.,2018,1,4
417,1629027,Trae Young,2018,1,5


## 2. Data Collection — Season Stats (1990–91 to 2022–23)
Pull total stats for every NBA player in every season. Cache to disk to avoid re-fetching.

In [4]:
from nba_api.stats.endpoints import leaguedashplayerstats

def season_str(year):
    return f'{year}-{str(year + 1)[-2:]}'

SEASONS    = [season_str(y) for y in range(1990, 2023)]
CACHE_PATH = OUTPUTS_DIR + 'raw_season_stats.csv'

if os.path.exists(CACHE_PATH):
    print(f'Loading cached stats from {CACHE_PATH}...')
    all_df = pd.read_csv(CACHE_PATH)
else:
    print(f'Pulling {len(SEASONS)} seasons from nba_api (this takes ~30s)...')
    frames = []
    for season in SEASONS:
        try:
            time.sleep(0.7)
            resp = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                per_mode_detailed='Totals',
                measure_type_detailed_defense='Base'
            )
            df = resp.get_data_frames()[0]
            df['SEASON']       = season
            df['SEASON_START'] = int(season.split('-')[0])
            frames.append(df)
            print(f'  {season}: {len(df)} players')
        except Exception as e:
            print(f'  {season}: FAILED — {e}')

    all_df = pd.concat(frames, ignore_index=True)
    all_df.to_csv(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

print(f'\nTotal season-player rows: {len(all_df):,}')
print(f'Seasons: {all_df["SEASON"].nunique()} | Players: {all_df["PLAYER_ID"].nunique():,}')

Pulling 33 seasons from nba_api (this takes ~30s)...
  1990-91: 0 players
  1991-92: 0 players
  1992-93: 0 players
  1993-94: 0 players
  1994-95: 0 players
  1995-96: 0 players
  1996-97: 441 players
  1997-98: 439 players
  1998-99: 440 players
  1999-00: 439 players
  2000-01: 441 players
  2001-02: 440 players
  2002-03: 428 players
  2003-04: 442 players
  2004-05: 464 players
  2005-06: 458 players
  2006-07: 458 players
  2007-08: 451 players
  2008-09: 445 players
  2009-10: 442 players
  2010-11: 452 players
  2011-12: 478 players
  2012-13: 469 players
  2013-14: 482 players
  2014-15: 492 players
  2015-16: 476 players
  2016-17: 486 players
  2017-18: 540 players
  2018-19: 530 players
  2019-20: 529 players
  2020-21: 540 players
  2021-22: 605 players
  2022-23: 539 players
Cached to outputs/raw_season_stats.csv

Total season-player rows: 12,846
Seasons: 27 | Players: 2,551


## 3. Data Joining & Cohort Construction
Join season stats to draft data on player_id. Compute career year, draft position bucket, and a composite performance score.

In [5]:
all_df['PLAYER_ID'] = all_df['PLAYER_ID'].astype('int64')

merged = all_df.merge(
    draft[['PERSON_ID', 'DRAFT_YEAR', 'ROUND_NUMBER', 'OVERALL_PICK']],
    left_on='PLAYER_ID', right_on='PERSON_ID', how='inner'
)

# Career year 1 = rookie season
merged['CAREER_YEAR'] = merged['SEASON_START'] - merged['DRAFT_YEAR'] + 1

# Keep only valid rows
merged = merged[
    (merged['CAREER_YEAR'] >= 1) &
    (merged['CAREER_YEAR'] <= MAX_CAREER) &
    (merged['GP']          >= MIN_GP)
].copy()

# Draft position bucket
def pick_bucket(p):
    if p <= 5:   return 'Top 5'
    if p <= 14:  return 'Lottery (6–14)'
    if p <= 30:  return 'Late 1st (15–30)'
    return '2nd Round'

merged['PICK_BUCKET'] = merged['OVERALL_PICK'].apply(pick_bucket)

# Composite performance score (per game): PTS + 1.2*REB + 1.5*AST + STL + 1.5*BLK - TOV
for col in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV']:
    merged[col + '_PG'] = merged[col] / merged['GP']

merged['PERF'] = (
    merged['PTS_PG']
    + 1.2 * merged['REB_PG']
    + 1.5 * merged['AST_PG']
    + 1.0 * merged['STL_PG']
    + 1.5 * merged['BLK_PG']
    - 1.0 * merged['TOV_PG']
).round(3)

print(f'Merged rows: {len(merged):,}')
print(f'Draft classes: {merged["DRAFT_YEAR"].nunique()} | Career years: 1–{merged["CAREER_YEAR"].max()}')
merged[['PLAYER_NAME', 'DRAFT_YEAR', 'SEASON', 'CAREER_YEAR', 'GP', 'PERF']].head(10)

TypeError: Expected numeric dtype, got object instead.

## 4. Cohort Matrices
Build two matrices — survival (% still active) and value (avg performance). Both in the exact format of a SaaS cohort retention table.

In [ ]:
# ── Survival: % of original class still playing ──────────────────────────
active_counts = (
    merged.groupby(['DRAFT_YEAR', 'CAREER_YEAR'])['PLAYER_ID']
    .nunique()
    .reset_index(name='N_ACTIVE')
)
active_counts = active_counts.merge(cohort_sizes.reset_index(), on='DRAFT_YEAR')
active_counts['PCT_ACTIVE'] = (active_counts['N_ACTIVE'] / active_counts['COHORT_SIZE'] * 100).round(1)

survival_matrix = active_counts.pivot(
    index='DRAFT_YEAR', columns='CAREER_YEAR', values='PCT_ACTIVE'
).sort_index()

# ── Value: avg performance per (draft_year, career_year) ─────────────────
value_stats = (
    merged.groupby(['DRAFT_YEAR', 'CAREER_YEAR'])['PERF']
    .mean()
    .reset_index(name='AVG_PERF')
)
value_matrix = value_stats.pivot(
    index='DRAFT_YEAR', columns='CAREER_YEAR', values='AVG_PERF'
).sort_index().round(2)

# ── LTV: total career contribution per class ──────────────────────────────
ltv_df = (
    merged.groupby('DRAFT_YEAR')
    .agg(
        N_PLAYERS=('PLAYER_ID', 'nunique'),
        TOTAL_PERF=('PERF', 'sum'),
        AVG_CAREER_YRS=('CAREER_YEAR', 'mean'),
    )
    .reset_index()
)
ltv_df['LTV_PER_PLAYER'] = (ltv_df['TOTAL_PERF'] / ltv_df['N_PLAYERS']).round(2)
ltv_df = ltv_df.sort_values('DRAFT_YEAR').reset_index(drop=True)

# Save
survival_matrix.to_csv(OUTPUTS_DIR + 'cohort_matrix.csv')
ltv_df.to_csv(OUTPUTS_DIR + 'draft_class_ltv.csv', index=False)

print('Survival matrix:')
print(survival_matrix.iloc[:5, :8])
print(f'\nTop 5 classes by LTV per player:')
print(ltv_df.nlargest(5, 'LTV_PER_PLAYER')[['DRAFT_YEAR', 'LTV_PER_PLAYER', 'N_PLAYERS']].to_string(index=False))

## 5. Cohort Retention Heatmap
The main visual. This is a SaaS retention table — rows are cohorts (draft years), columns are time periods (career years), cells are % retained.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 13))

sns.heatmap(
    survival_matrix,
    cmap='YlOrRd',
    annot=True,
    fmt='.0f',
    linewidths=0.4,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': '% of draft class still active (≥20 GP)', 'shrink': 0.55},
    annot_kws={'size': 7},
    vmin=0, vmax=80
)

ax.set_title(
    'NBA Draft Class Retention Grid  (1990–2018)\n'
    'Each cell = % of that draft class still playing ≥20 games in that career year — identical format to a SaaS monthly cohort table',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_xlabel('Career Year  (1 = rookie season)', fontsize=11)
ax.set_ylabel('Draft Year  (Cohort)', fontsize=11)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'cohort_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cohort_heatmap.png')

## 6. LTV Analysis — Best & Worst Draft Classes

In [ ]:
HIGHLIGHT = [2003, 1996, 1984, 2011, 1992]   # known great classes

ltv_sorted = ltv_df.sort_values('LTV_PER_PLAYER', ascending=True).reset_index(drop=True)
colors_bar = [
    '#9B3D36' if yr in HIGHLIGHT
    else '#4C72B0' if ltv_sorted.loc[ltv_sorted['DRAFT_YEAR']==yr,'LTV_PER_PLAYER'].values[0] >= ltv_df['LTV_PER_PLAYER'].median()
    else '#B0B8C4'
    for yr in ltv_sorted['DRAFT_YEAR']
]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('NBA Draft Class LTV Analysis  (1990–2018)', fontsize=13, fontweight='bold')

# Left — LTV bar chart
ax = axes[0]
bars = ax.barh(ltv_sorted['DRAFT_YEAR'].astype(str), ltv_sorted['LTV_PER_PLAYER'],
               color=colors_bar, alpha=0.88)
ax.axvline(ltv_df['LTV_PER_PLAYER'].median(), color='black', linestyle='--',
           linewidth=1.2, alpha=0.5, label='Median')
ax.set_title('LTV Per Player by Draft Class\n(Total career performance score ÷ players in class)', fontsize=10)
ax.set_xlabel('Avg Career Performance Score Per Player')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25, axis='x')
for bar, val in zip(bars, ltv_sorted['LTV_PER_PLAYER']):
    ax.text(val * 1.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=7)

# Right — Trend scatter
ax2 = axes[1]
scatter_colors = ['#9B3D36' if yr in HIGHLIGHT else '#4C72B0' for yr in ltv_df['DRAFT_YEAR']]
ax2.scatter(ltv_df['DRAFT_YEAR'], ltv_df['LTV_PER_PLAYER'],
            color=scatter_colors, s=65, zorder=3, alpha=0.85)
m, b = np.polyfit(ltv_df['DRAFT_YEAR'], ltv_df['LTV_PER_PLAYER'], 1)
x_line = np.linspace(1990, 2018, 100)
ax2.plot(x_line, m * x_line + b, color='#DD8452', linestyle='--', linewidth=1.5, alpha=0.8, label='Trend')
for _, row in ltv_df.iterrows():
    if row['DRAFT_YEAR'] in HIGHLIGHT + [ltv_df.nsmallest(3,'LTV_PER_PLAYER')['DRAFT_YEAR'].tolist()[0]]:
        ax2.annotate(str(int(row['DRAFT_YEAR'])),
                     (row['DRAFT_YEAR'], row['LTV_PER_PLAYER']),
                     textcoords='offset points', xytext=(5, 3), fontsize=8)
ax2.set_title('Draft Class LTV Over Time — Is the League Getting Deeper?', fontsize=10)
ax2.set_xlabel('Draft Year')
ax2.set_ylabel('LTV Per Player')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'draft_class_ltv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: draft_class_ltv.png')

## 7. Statistical Testing
1. **ANOVA + Kruskal-Wallis** — are cohort LTV differences statistically real or noise?  
2. **Trend test** — are more recent drafts better or worse?

In [ ]:
# Group performance values by draft year (individual player-seasons)
groups = [
    merged[merged['DRAFT_YEAR'] == yr]['PERF'].dropna().values
    for yr in sorted(merged['DRAFT_YEAR'].unique())
    if len(merged[merged['DRAFT_YEAR'] == yr]) >= 15
]

f_stat, p_anova    = f_oneway(*groups)
h_stat, p_kruskal  = kruskal(*groups)

print('=== Cohort Quality Differences ===')
print(f'One-way ANOVA:       F={f_stat:.2f}  p={p_anova:.6f}  →  {"SIGNIFICANT" if p_anova < 0.05 else "not significant"}')
print(f'Kruskal-Wallis:      H={h_stat:.2f}  p={p_kruskal:.6f}  →  {"SIGNIFICANT" if p_kruskal < 0.05 else "not significant"}')
print('Draft class quality differences are statistically real, not random variation.\n' if p_anova < 0.05 else 'Cannot reject null — differences may be noise.\n')

# Trend: is draft quality improving over time?
r_p, p_p = pearsonr(ltv_df['DRAFT_YEAR'], ltv_df['LTV_PER_PLAYER'])
r_s, p_s = spearmanr(ltv_df['DRAFT_YEAR'], ltv_df['LTV_PER_PLAYER'])
print('=== LTV Trend Over Time ===')
print(f'Pearson r  = {r_p:.3f}  p={p_p:.4f}')
print(f'Spearman r = {r_s:.3f}  p={p_s:.4f}')
direction = 'IMPROVING' if r_p > 0 else 'DECLINING'
sig = 'significant' if p_p < 0.05 else 'not significant'
print(f'Trend: draft class LTV is {direction} over time ({sig})')

# Average survival rates at key career milestones
print('\n=== Average Cohort Survival Rates ===')
for yr in [1, 3, 5, 8, 10, 12]:
    col = yr
    if col in survival_matrix.columns:
        avg = survival_matrix[col].mean()
        print(f'  Career Year {yr:>2}: {avg:.1f}% still active on average')

## 8. Draft Position Survival Curves
The draft position equivalent of acquisition channel analysis — which channel produces the best long-term retention?

In [ ]:
PICK_ORDER  = ['Top 5', 'Lottery (6–14)', 'Late 1st (15–30)', '2nd Round']
PICK_COLORS = {'Top 5': '#9B3D36', 'Lottery (6–14)': '#4C72B0',
               'Late 1st (15–30)': '#55A868', '2nd Round': '#DD8452'}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Career Survival by Draft Position', fontsize=13, fontweight='bold')

# Left: survival curves
ax = axes[0]
for bucket in PICK_ORDER:
    sub   = merged[merged['PICK_BUCKET'] == bucket]
    total = sub['PLAYER_ID'].nunique()
    if total == 0:
        continue
    survival = sub.groupby('CAREER_YEAR')['PLAYER_ID'].nunique() / total * 100
    ax.plot(survival.index, survival.values, marker='o', markersize=4, linewidth=2,
            color=PICK_COLORS[bucket], label=f'{bucket}  (n={total})', alpha=0.85)

ax.set_title('% of group still active by career year', fontsize=10)
ax.set_xlabel('Career Year')
ax.set_ylabel('% Still Active (≥20 GP)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: bar chart — % reaching career year 10
ax2 = axes[1]
reach_yr10 = {}
for bucket in PICK_ORDER:
    sub    = merged[merged['PICK_BUCKET'] == bucket]
    total  = sub['PLAYER_ID'].nunique()
    active = sub[sub['CAREER_YEAR'] >= 10]['PLAYER_ID'].nunique()
    reach_yr10[bucket] = round(active / total * 100, 1) if total > 0 else 0

ax2.bar(reach_yr10.keys(), reach_yr10.values(),
        color=[PICK_COLORS[b] for b in reach_yr10], alpha=0.85, width=0.5)
ax2.set_title('% Reaching Career Year 10', fontsize=10)
ax2.set_ylabel('%')
ax2.grid(True, alpha=0.3, axis='y')
for i, (bucket, val) in enumerate(reach_yr10.items()):
    ax2.text(i, val + 0.5, f'{val}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'survival_by_draft_position.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: survival_by_draft_position.png')

## 9. Top Players Per Draft Class
Sanity check: do the right names come out? Also used for the LinkedIn post and article.

In [ ]:
player_career = (
    merged.groupby(['PLAYER_ID', 'PLAYER_NAME', 'DRAFT_YEAR'])['PERF']
    .sum()
    .reset_index(name='CAREER_PERF')
    .sort_values('CAREER_PERF', ascending=False)
)

top3 = (
    player_career
    .groupby('DRAFT_YEAR')
    .head(3)
    .sort_values(['DRAFT_YEAR', 'CAREER_PERF'], ascending=[True, False])
)

top5_classes = ltv_df.nlargest(5, 'LTV_PER_PLAYER')['DRAFT_YEAR'].tolist()
bot5_classes = ltv_df.nsmallest(5, 'LTV_PER_PLAYER')['DRAFT_YEAR'].tolist()

print('=== Top 5 Draft Classes by LTV Per Player ===')
for yr in sorted(top5_classes):
    ltv_val = ltv_df.loc[ltv_df['DRAFT_YEAR'] == yr, 'LTV_PER_PLAYER'].values[0]
    names   = top3[top3['DRAFT_YEAR'] == yr]['PLAYER_NAME'].tolist()
    print(f'  {yr}  LTV={ltv_val:.1f}  |  {"  ·  ".join(names)}')

print('\n=== Bottom 5 Draft Classes by LTV Per Player ===')
for yr in sorted(bot5_classes):
    ltv_val = ltv_df.loc[ltv_df['DRAFT_YEAR'] == yr, 'LTV_PER_PLAYER'].values[0]
    names   = top3[top3['DRAFT_YEAR'] == yr]['PLAYER_NAME'].tolist()
    print(f'  {yr}  LTV={ltv_val:.1f}  |  {"  ·  ".join(names)}')

print('\n=== 2003 Class (The Outlier) ===')
class_2003 = player_career[player_career['DRAFT_YEAR'] == 2003].head(10)
print(class_2003[['PLAYER_NAME', 'CAREER_PERF']].to_string(index=False))

## 10. Performance Value Heatmap
Same grid but showing average performance score (not just survival %). Reveals which classes produced higher quality players, not just more survivors.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 13))

sns.heatmap(
    value_matrix,
    cmap='Blues',
    annot=True,
    fmt='.1f',
    linewidths=0.4,
    linecolor='white',
    ax=ax,
    cbar_kws={
        'label': 'Avg Performance Score per game  (PTS + 1.2·REB + 1.5·AST + STL + 1.5·BLK – TOV)',
        'shrink': 0.55
    },
    annot_kws={'size': 7}
)
ax.set_title(
    'NBA Draft Class — Average Player Performance by Career Year  (1990–2018)\n'
    'Darker = higher average contribution among players still active that year',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_xlabel('Career Year', fontsize=11)
ax.set_ylabel('Draft Year (Cohort)', fontsize=11)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR + 'value_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: value_heatmap.png')

## 11. Key Findings

In [ ]:
print('=' * 60)
print('KEY FINDINGS — NBA Draft Cohort Analysis 1990-2018')
print('=' * 60)

best  = ltv_df.nlargest(1,  'LTV_PER_PLAYER').iloc[0]
worst = ltv_df.nsmallest(1, 'LTV_PER_PLAYER').iloc[0]

print(f'\nDraft classes:        {ltv_df["DRAFT_YEAR"].nunique()}')
print(f'Player-seasons used:  {len(merged):,}')
print(f'Unique players:       {merged["PLAYER_ID"].nunique():,}')

print(f'\nBest class:   {int(best.DRAFT_YEAR)}  (LTV={best.LTV_PER_PLAYER:.1f})')
print(f'Worst class:  {int(worst.DRAFT_YEAR)}  (LTV={worst.LTV_PER_PLAYER:.1f})')
print(f'Gap:          {(best.LTV_PER_PLAYER / worst.LTV_PER_PLAYER):.1f}x')

print(f'\nStatistical test:')
print(f'  ANOVA: F={f_stat:.2f}  p={p_anova:.6f}  → {"significant" if p_anova < 0.05 else "not significant"}')

print(f'\nDraft position — % reaching career year 10:')
for bucket, val in reach_yr10.items():
    print(f'  {bucket:<22} {val:.1f}%')

print(f'\nAverage class survival:')
for yr in [3, 5, 8, 10]:
    if yr in survival_matrix.columns:
        print(f'  Year {yr}: {survival_matrix[yr].mean():.1f}% still active')